# U.S. County Home Values | Regression Models
**Question:** How well do county-level income, population, and education indicators predict median owner-occupied home values?

**Source:** U.S. Census Bureau, 2023 ACS five-year estimates. This notebook compares a mean baseline, linear regression, and random forest.

**Before running:** You will be prompted once for your Census API key. Paste it into the hidden prompt, **not into the code**. Do not publish notebook outputs containing credentials. Previously published model scores must be recalculated after data cleaning.

**Security:** This notebook never stores an API key in its source. Enter your key only when prompted at runtime. Do not save notebook outputs containing credentials.

## 1. Authenticate and prepare data
Run all cells from the top. The first code cell asks for a Census API key once per Python session. It verifies access using a small North Carolina query before downloading national data.

Install dependencies if needed: `pip install pandas numpy requests matplotlib scikit-learn`.

In [ ]:
import os
import getpass
import requests

if not os.environ.get("CENSUS_API_KEY", "").strip():
    os.environ["CENSUS_API_KEY"] = getpass.getpass("Enter your Census API key (hidden): ").strip()

if not os.environ["CENSUS_API_KEY"]:
    raise ValueError("A Census API key is required for this data refresh.")

check = requests.get(
    "https://api.census.gov/data/2023/acs/acs5",
    params={"get": "NAME,B19013_001E", "for": "county:*", "in": "state:37",
            "key": os.environ["CENSUS_API_KEY"]},
    timeout=30, allow_redirects=False,
)
if check.status_code != 200 or not check.text.lstrip().startswith("["):
    raise RuntimeError(
        f"Census authentication failed (HTTP {check.status_code}; "
        f"redirect: {check.headers.get('Location', 'none')}). "
        "Confirm your key is activated and retry."
    )
print("Census API key accepted; proceeding with 2023 ACS data.")

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

REFRESH_FROM_CENSUS = True
DATA_CANDIDATES = [Path("../data/county_data_clean.csv"), Path("data/county_data_clean.csv"), Path("county_data_clean.csv")]
OUTPUT_DIR = Path("../assets") if Path("../assets").is_dir() else Path("assets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def fetch_acs():
    """Download 2023 ACS 5-year county estimates with transparent error handling."""
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    url = "https://api.census.gov/data/2023/acs/acs5"
    fields = [
        "NAME", "B01001_001E", "B19013_001E",
        "B25077_001E", "B15003_022E", "B15003_001E",
    ]
    params = {
        "get": ",".join(fields),
        "for": "county:*",
        "in": "state:*",
    }
    key = os.getenv("CENSUS_API_KEY")
    if key:
        params["key"] = key

    session = requests.Session()
    retry = Retry(
        total=3, backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"Accept": "application/json", "User-Agent": "portfolio-research-notebook/1.0"})

    def request_table(query):
        try:
            response = session.get(url, params=query, timeout=90, allow_redirects=False)
            response.raise_for_status()
            if response.is_redirect or "missing_key.html" in response.url:
                raise RuntimeError("Census API redirected the request; verify the API key.")
        except requests.RequestException as exc:
            detail = ""
            if getattr(exc, "response", None) is not None:
                detail = f" HTTP {exc.response.status_code}; body: {exc.response.text[:250]!r}"
            raise RuntimeError(f"Census API request failed.{detail} URL: {url}") from exc
        try:
            rows = response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"Census API returned non-JSON content (HTTP {response.status_code}; "
                f"content type: {response.headers.get('Content-Type', 'unknown')}). "
                f"Response preview: {response.text[:300]!r}. "
                "Try again later or check https://api.census.gov/data/2023/acs/acs5.html"
            ) from exc
        if not isinstance(rows, list) or len(rows) < 2:
            raise RuntimeError(f"Unexpected Census API payload: {str(rows)[:300]}")
        return pd.DataFrame(rows[1:], columns=rows[0])

    try:
        return request_table(params)
    except RuntimeError as exc:
        print(f"Combined county request failed: {exc}")
        print("Trying separate state-level requests...")

    # Fallback can avoid temporary errors on a large nationwide response.
    frames = []
    failed = []
    for state_code in ["01", "02", "04", "05", "06", "08", "09", "10", "11", "12", "13", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31", "32", "33", "34", "35", "36", "37", "38", "39", "40", "41", "42", "44", "45", "46", "47", "48", "49", "50", "51", "53", "54", "55", "56", "72"]:
        state_params = dict(params)
        state_params["in"] = f"state:{state_code}"
        try:
            frame = request_table(state_params)
            frames.append(frame)
        except RuntimeError:
            failed.append(state_code)
    if not frames or failed:
        raise RuntimeError(
            f"Census download incomplete. Failed state FIPS codes: {failed}. "
            "No partial dataset was used. Check API connectivity and rerun."
        )
    return pd.concat(frames, ignore_index=True)

csv_path = next((p for p in DATA_CANDIDATES if p.is_file()), None)
if csv_path and not REFRESH_FROM_CENSUS:
    source = pd.read_csv(csv_path)
    print(f"Loaded {len(source)} rows from {csv_path}")
else:
    source = fetch_acs()
    print(f"Downloaded {len(source)} ACS records")

rename = {
    "B01001_001E": "Total_Population",
    "B19013_001E": "Median_Income",
    "B25077_001E": "Median_Home_Value",
    "B15003_022E": "Bachelors_Count",
    "B15003_001E": "Population_25_Plus",
}
source = source.rename(columns=rename)
required = ["Median_Income", "Median_Home_Value", "Total_Population", "Bachelors_Count", "Population_25_Plus"]
if "Population_25_Plus" not in source.columns or "Bachelors_Count" not in source.columns:
    print("Existing CSV lacks the age-25+ denominator; downloading ACS source data.")
    source = fetch_acs().rename(columns=rename)
missing = set(required) - set(source.columns)
if missing:
    raise ValueError(f"Missing ACS variables: {sorted(missing)}")

for column in required:
    source[column] = pd.to_numeric(source[column], errors="coerce")
# Census negative sentinel values indicate unavailable estimates, not valid measurements.
source[required] = source[required].mask(source[required] < 0)
before = len(source)
df = source.dropna(subset=required).copy()
df = df[
    (df["Median_Income"] > 0) &
    (df["Median_Home_Value"] > 0) &
    (df["Total_Population"] > 0) &
    (df["Population_25_Plus"] > 0) &
    (df["Bachelors_Count"] >= 0) &
    (df["Bachelors_Count"] <= df["Population_25_Plus"])
].copy()
df["Bachelors_Rate"] = df["Bachelors_Count"] / df["Population_25_Plus"]
if len(df) < 30:
    raise ValueError("Too few valid rows after cleaning; inspect source data.")
print(f"Rows retained: {len(df)} of {before}; excluded: {before - len(df)}")
display(df[["Median_Income", "Median_Home_Value", "Total_Population", "Bachelors_Rate"]].describe().round(2))

## 2. Exploratory analysis
Correlation describes association in this dataset, not causation. The cleaned records may differ from the original 3,216-row sample.

In [ ]:
features = ["Median_Income", "Total_Population", "Bachelors_Rate"]
target = "Median_Home_Value"
corr = df[[target] + features].corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
labels = ["Home value", "Income", "Population", "Bachelor's rate"]
ax.set_xticks(range(4), labels, rotation=25, ha="right")
ax.set_yticks(range(4), labels)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.55 else "black")
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("County-level variable correlations")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "housing-correlation.png", dpi=170)
plt.show()

## 3. Train/test split and models
A fixed 80/20 random split allows consistent comparisons. Standardization is fitted **only on training data** for linear regression; the tree model uses raw features. A geographic holdout would provide a stronger test of generalization to new regions.

In [ ]:
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random forest": RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
    ),
}
predictions = {}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    })
results = pd.DataFrame(rows)
display(results.round(3))
print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")

## 4. Model comparison and diagnostics
Lower RMSE and MAE indicate smaller prediction errors; higher R² indicates more explained variation on the held-out test set. These metrics apply to this split and dataset, not individual home appraisals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(results["Model"], results["RMSE"], color="#3b82f6")
axes[0].set_title("Test RMSE (lower is better)")
axes[0].set_ylabel("US dollars")
axes[1].bar(results["Model"], results["R2"], color="#14b8a6")
axes[1].set_title("Test R² (higher is better)")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "housing-models.png", dpi=170)
plt.show()

In [ ]:
forest = models["Random forest"]
importances = pd.Series(forest.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importances.index, importances.values, color="#14b8a6")
ax.set_title("Random forest feature importance")
ax.set_xlabel("Impurity-based relative importance")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "housing-importance.png", dpi=170)
plt.show()

In [ ]:
pred = predictions["Random forest"]
residuals = y_test.to_numpy() - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, pred, s=16, alpha=0.6)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(limits, limits, linestyle="--", color="gray")
axes[0].set(xlabel="Actual home value ($)", ylabel="Predicted ($)",
            title="Actual vs. predicted")
axes[1].scatter(pred, residuals, s=16, alpha=0.6)
axes[1].axhline(0, linestyle="--", color="gray")
axes[1].set(xlabel="Predicted ($)", ylabel="Residual ($)",
            title="Residuals")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "housing-predictions.png", dpi=170)
plt.show()

## 5. Save validated artifacts
Exports below overwrite the *local* CSV files only after successful cleaning and model evaluation. Review them before replacing files already published in GitHub.

In [ ]:
EXPORT_DIR = Path("../data") if Path("../data").is_dir() else Path("data")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPORT_DIR / "county_data_clean.csv", index=False)
results.to_csv(EXPORT_DIR / "model_comparison.csv", index=False)
print(f"Saved cleaned data and model comparison to {EXPORT_DIR.resolve()}")

## 6. Interpretation and limitations
- Compare the three models using the **newly generated** table, not the earlier reported scores.
- Census negative sentinels are excluded; bachelor's attainment is calculated using the population age 25 and older.
- A county-level median is not an individual property price. Aggregate relationships do not imply causation.
- A single random split does not establish stability across regions or years.
- Impurity-based feature importance describes how the fitted forest used variables; it does not identify causal drivers.

**Next step:** Test geographic holdouts and additional predictors after verifying the refreshed ACS dataset.

**Results from the October 2026 run (80/20 split; 2,572 training, 643 test rows):** mean baseline RMSE $141,021 / R² −0.006; linear regression RMSE $83,688 / R² 0.646; random forest RMSE $72,317 / R² 0.735. These are historical run results; reruns may differ if source data or dependencies change.